# Bronze Ingestion (Secure + Streaming)

## Fetch Secrets from Key Vault

In [0]:
client_id = dbutils.secrets.get(scope="vedica_new", key="client-id")
client_secret = dbutils.secrets.get(scope="vedica_new", key="client-secret")
tenant_id = dbutils.secrets.get(scope="vedica_new", key="tenant-id")

## Configure ADLS Gen2 OAuth

In [0]:
storage_account = "stvedica2411"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

## Event Hub Config

In [0]:
event_hub_connection_string = dbutils.secrets.get(scope="vedica_new", key="connection-string")
event_hub_name = "order-stream"

In [0]:
encrypted_cs = sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(event_hub_connection_string)

eh_conf = {
  "eventhubs.connectionString": encrypted_cs
}

## Define Bronze Paths

In [0]:
bronze_path = "abfss://bronze@stvedica2411.dfs.core.windows.net/orders"
checkpoint_path = "abfss://bronze@stvedica2411.dfs.core.windows.net/checkpoints/orders_bronze"

## Define Schema

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("timestamp", StringType(), True),
    StructField("region", StringType(), True)
])

## Read Stream from Event Hub

In [0]:
raw_stream_df = (
    spark.readStream
         .format("eventhubs")
         .options(**eh_conf)
         .load()
)

## Parse JSON Body

In [0]:
from pyspark.sql.functions import col, from_json

json_df = (
    raw_stream_df
    .selectExpr("CAST(body AS STRING) as json_data")
    .select(from_json(col("json_data"), schema).alias("data"))
    .select("data.*")
)

## Add Bronze Metadata

In [0]:
from pyspark.sql.functions import current_timestamp, to_date

bronze_df = (
    json_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("event_date", to_date(col("timestamp")))
)

## Write to Bronze Delta

In [0]:
(
    bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .partitionBy("event_date")
        .trigger(processingTime="2 minutes")
        .start(bronze_path)
)

# ## ### **🥈 Silver Transformation Code (Databricks – PySpark)
**

# Define Paths


In [0]:
bronze_path = "abfss://bronze@stvedica2411.dfs.core.windows.net/orders"
silver_path = "abfss://silver@stvedica2411.dfs.core.windows.net/orders"
checkpoint_path_silver = "abfss://silver@stvedica2411.dfs.core.windows.net/checkpoints/orders_silver"

# ## Read Bronze as Stream


In [0]:
bronze_df = (
    spark.readStream
         .format("delta")
         .load(bronze_path)
)

# ## Clean / Transform Data

In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import IntegerType, DoubleType, TimestampType



## # ## Handle Nulls

In [0]:
clean_df = bronze_df.dropna(subset=["order_id", "customer_id", "price", "quantity"])

## Deduplicate on order_id

In [0]:
dedup_df = clean_df.dropDuplicates(["order_id"])

## Cast Data Types

In [0]:
typed_df = (
    dedup_df
    .withColumn("order_id", col("order_id").cast(IntegerType()))
    .withColumn("quantity", col("quantity").cast(IntegerType()))
    .withColumn("price", col("price").cast(DoubleType()))
    .withColumn("timestamp", col("timestamp").cast(TimestampType()))
)

## Add ingestion timestamp


In [0]:
silver_df = typed_df.withColumn("ingestion_timestamp", current_timestamp())

## Write to Silver Delta

In [0]:
(
    silver_df.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", checkpoint_path_silver)
             .start(silver_path)
)

# 🥇 Gold Layer

## Paths (Adjust Once)


In [0]:
silver_path = "abfss://silver@stvedica2411.dfs.core.windows.net/orders"

gold_revenue_path = "abfss://gold@stvedica2411.dfs.core.windows.net/revenue_by_region"
gold_top_products_path = "abfss://gold@stvedica2411.dfs.core.windows.net/top_products"
gold_clv_path = "abfss://gold@stvedica2411.dfs.core.windows.net/customer_clv"

## Read Silver Data (Streaming)


In [0]:
silver_df = (
    spark.readStream
         .format("delta")
         .load(silver_path)
)

## Total Revenue per Region per Day

In [0]:
from pyspark.sql.functions import sum, col

revenue_df = (
    silver_df
    .withColumn("revenue", col("quantity") * col("price"))
    .groupBy("region", "event_date")
    .agg(sum("revenue").alias("total_revenue"))
)

## Write to Gold

In [0]:
(
    revenue_df.writeStream
              .format("delta")
              .outputMode("complete")
              .option("checkpointLocation", "abfss://gold@stvedica2411.dfs.core.windows.net/checkpoints/revenue")
              .start(gold_revenue_path)
)

## Top 10 Products by Quantity per Week


## Streaming Aggregation (Silver → Weekly Totals)

In [0]:
from pyspark.sql.functions import col, sum, weekofyear, year

silver_path = "abfss://silver@stvedica2411.dfs.core.windows.net/orders"
gold_weekly_path = "abfss://gold@stvedica2411.dfs.core.windows.net/weekly_product_sales"
checkpoint_path = "abfss://gold@stvedica2411.dfs.core.windows.net/checkpoints/weekly_product_sales"

# Read Silver as STREAM
silver_df = spark.readStream.format("delta").load(silver_path)

weekly_sales_df = (
    silver_df
    .withColumn("week", weekofyear(col("timestamp")))
    .withColumn("year", year(col("timestamp")))
    .groupBy("year", "week", "product_id")
    .agg(sum("quantity").alias("total_quantity"))
)

query = (
    weekly_sales_df.writeStream
    .format("delta")
    .outputMode("complete")   # aggregation requires complete/update
    .option("checkpointLocation", checkpoint_path)
    .start(gold_weekly_path)
)

## Batch Ranking (Top 10 Products)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

gold_weekly_path = "abfss://gold@stvedica2411.dfs.core.windows.net/weekly_product_sales"

weekly_sales_batch_df = spark.read.format("delta").load(gold_weekly_path)

window_spec = Window.partitionBy("year", "week").orderBy(col("total_quantity").desc())

top_products_df = (
    weekly_sales_batch_df
    .withColumn("rank", dense_rank().over(window_spec))
    .filter(col("rank") <= 10)
)

top_products_df.display()

year,week,product_id,total_quantity,rank


## Customer Lifetime Value (CLV) – Window Function

## CLV = running total of customer spend

## Read Silver Table (Batch)

In [0]:
from pyspark.sql.functions import col

silver_path = "abfss://silver@stvedica2411.dfs.core.windows.net/orders"

silver_df = spark.read.format("delta").load(silver_path)

## Calculate Revenue

In [0]:
from pyspark.sql.functions import expr

revenue_df = silver_df.withColumn(
    "revenue",
    expr("quantity * price")
)

## Apply Window Function (CLV)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy("timestamp")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

clv_df = revenue_df.withColumn(
    "clv",
    sum("revenue").over(window_spec)
)

## Display Result

In [0]:
clv_df.select(
    "customer_id",
    "order_id",
    "timestamp",
    "revenue",
    "clv"
).display()

customer_id,order_id,timestamp,revenue,clv


## Save to Gold Layer

In [0]:
gold_clv_path = "abfss://gold@stvedica2411.dfs.core.windows.net/customer_clv"

clv_df.write.format("delta").mode("overwrite").save(gold_clv_path)